# 원강(Won-gang) 프로젝트: GPT-1 기반 챗봇 생성 모델 구현

## 1. GPT와 기존 Transformer 모델의 아키텍처 비교 및 변경 사항

본 프로젝트는 첨부된 논문 "Improving Language Understanding by Generative Pre-Training"을 기반으로 GPT 모델을 구현합니다. 기존 Transformer(Attention is all you need) 모델과 비교하여 GPT 아키텍처가 가지는 핵심적인 차이점은 다음과 같습니다.

1. **Decoder-Only 아키텍처**: 
   [cite_start]기존 Transformer가 인코더(Encoder)와 디코더(Decoder)를 모두 사용하는 구조였다면, GPT는 인코더를 완전히 제거하고 12개의 레이어를 가진 디코더(Decoder-only) 구조만을 사용합니다[cite: 177, 178]. 따라서 인코더-디코더 간의 Attention(Cross-Attention) 블록이 삭제됩니다.
   
2. **Masked Self-Attention의 활용**:
   [cite_start]입력된 컨텍스트 토큰에 대해 Multi-headed self-attention 연산을 적용하며, 이때 미래의 토큰을 보지 못하도록 마스킹(Masking)을 적용합니다[cite: 178, 179].
   
3. **학습 가능한 위치 임베딩 (Learned Positional Embedding)**:
   [cite_start]기존 Transformer가 사인/코사인 함수를 이용한 정적(Sinusoidal) 위치 인코딩을 사용한 반면, GPT는 모델이 학습 과정에서 위치 정보를 스스로 최적화하는 학습 가능한 위치 임베딩(Learned position embeddings)을 사용합니다[cite: 187]. [cite_start]초기 입력은 $h_{0}=UW_{e}+W_{p}$ 수식을 통해 토큰 임베딩 행렬($W_{e}$)과 위치 임베딩 행렬($W_{p}$)의 합으로 구성됩니다[cite: 80, 84].

4. **활성화 함수 (Activation Function)**:
   [cite_start]Feed-Forward 네트워크의 활성화 함수로 ReLU 대신 GELU(Gaussian Error Linear Unit)를 사용합니다[cite: 186].

In [3]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 32.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 58.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: protobuf90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/19 [termcolor]
    Found existing installation: protobuf 5.29.3━━━━━━━━━━━━━━  4/19 [termcolor]
    Uninstalling protobuf-5.29.3:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/19 [termcolor]
      Successfully uninstalled protobuf-5.29.3━━━━━━━━━━━━━━━━━━━━  5/19 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]9 [tensorflow]-py]


In [4]:
# 1. 필요 라이브러리 임포트
import urllib.request
import pandas as pd
import tensorflow as tf
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 데이터 다운로드 (songys/Chatbot_data)
urllib.request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatbotData.csv")
train_data = pd.read_csv('ChatbotData.csv')

# 데이터 구조 확인 및 샘플링 (학습 속도를 위해 일부만 사용)
train_data = train_data[:2000] 

# 과제 2: GPT 모델의 입력 형태에 맞게 전처리 수행 (Decoder 기반 생성 모델)
# 구조화된 입력은 연속적인 토큰 시퀀스로 변환되어야 합니다[cite: 149].
# Q와 A를 구분자(Delimiter)로 연결하여 단일 시퀀스로 만듭니다[cite: 164].
# 형태: <BOS> Question <SEP> Answer <EOS>

BOS = "<BOS> "
SEP = " <SEP> "
EOS = " <EOS>"

def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    return sentence.strip()

# Decoder 입력용 데이터 생성
sequences = []
for _, row in train_data.iterrows():
    q = preprocess_sentence(row['Q'])
    a = preprocess_sentence(row['A'])
    # 논문의 input transformation 방식에 따른 시퀀스 결합
    sequence = BOS + q + SEP + a + EOS
    sequences.append(sequence)

# 토크나이저 설정
tokenizer = Tokenizer(filters='', oov_token='<OOV>')
tokenizer.fit_on_texts(sequences)
vocab_size = len(tokenizer.word_index) + 1

# 정수 인코딩 및 패딩
sequences_int = tokenizer.texts_to_sequences(sequences)
max_len = max([len(seq) for seq in sequences_int])
padded_seqs = pad_sequences(sequences_int, maxlen=max_len, padding='post')

# Language Modeling을 위한 X(입력), Y(타겟) 분리
# GPT는 이전 토큰들로 다음 토큰을 예측하므로, Y는 X를 한 칸씩 뒤로 미룬 형태입니다.
X_train = padded_seqs[:, :-1]
y_train = padded_seqs[:, 1:]

print(f"Vocab Size: {vocab_size}")
print(f"Input Shape: {X_train.shape}, Target Shape: {y_train.shape}")

I0000 00:00:1782195811.683866      88 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782195811.803043      88 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782195814.738061      88 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Vocab Size: 4656
Input Shape: (2000, 20), Target Shape: (2000, 20)


In [6]:
# 과제 3 & 4: 모델의 입력 블록을 GPT 논문에 기반하여 수정 및 GPT 모델 구성
import tensorflow as tf

# 논문에 따라 학습 가능한 위치 임베딩(Learned Positional Embeddings)을 구현합니다.
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        # 1. 토큰 임베딩 (We)
        self.token_emb = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        # 2. 위치 임베딩 (Wp) - Sinusoidal 대신 학습 가능한 임베딩 사용
        self.pos_emb = tf.keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        max_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        # 수식 h_0 = U*We + Wp 구현
        return self.token_emb(x) + self.pos_emb(positions)

# 미래 토큰 마스킹 함수 (Causal Mask)
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    return tf.tile(mask, [batch_size, 1, 1])

# Transformer Decoder Block (GPT 아키텍처의 핵심)
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        # 변경사항: Masked Self Attention 적용
        self.att = tf.keras.layers.MultiHeadAttention(num_heads, embed_dim)
        # 변경사항: 활성화 함수로 GELU 사용
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation=tf.keras.activations.gelu), 
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    # 수정된 부분: training=False 기본값 추가
    def call(self, inputs, training=False):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        
        causal_mask = causal_attention_mask(batch_size, seq_len, seq_len, tf.bool)
        
        # Masked Multi-Head Self Attention
        attention_output = self.att(inputs, inputs, attention_mask=causal_mask)
        attention_output = self.dropout1(attention_output, training=training)
        out1 = self.layernorm1(inputs + attention_output)
        
        # Feed Forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# 모델 구성 파라미터 (빠른 학습을 위해 논문보다 작게 설정)
embed_dim = 128
num_heads = 4
ff_dim = 256
num_layers = 2 # 원 논문은 12개 층 이나 리소스 한계상 2개로 조정

# GPT 모델 빌드
inputs = tf.keras.layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)(inputs)

for _ in range(num_layers):
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

outputs = tf.keras.layers.Dense(vocab_size, activation="softmax")(x)

gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)
gpt_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                  loss="sparse_categorical_crossentropy")

# 과제 4: 모델 요약 및 훈련 결과 
gpt_model.summary()

# 훈련 실행 (결과 캡처용)
print("모델 학습을 시작합니다. 잠시만 기다려주세요...")
history = gpt_model.fit(X_train, y_train, batch_size=32, epochs=5)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, None, 128)      │       598,656 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, None, 128)      │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, None, 128)      │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, None, 4656)     │       600,624 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,859,760 (7.09 MB)

 Trainable params: 1,859,760 (7.09 MB)

 Non-trainable params: 0 (0.00 B)

모델 학습을 시작합니다. 잠시만 기다려주세요...
Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 28s 313ms/step - loss: 4.1616
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 318ms/step - loss: 2.4486
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 322ms/step - loss: 2.0666
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 315ms/step - loss: 1.7615
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 314ms/step - loss: 1.5081


In [8]:
# 과제 5: 출력 결과물 생성 (Autoregressive Generation) - 인덱스 오류 수정본

def generate_text(model, tokenizer, max_len, input_sentence):
    # 입력을 <BOS> 질문 <SEP> 형태로 구성
    q = preprocess_sentence(input_sentence)
    test_seq = f"{BOS}{q}{SEP}"
    
    encoded = tokenizer.texts_to_sequences([test_seq])[0]
    
    # 모델이 허용하는 최대 입력 길이(max_len - 1)까지만 생성하도록 반복 횟수 제한
    max_generate_steps = (max_len - 1) - len(encoded)
    
    for _ in range(max_generate_steps):
        padded = pad_sequences([encoded], maxlen=max_len-1, padding='post')
        
        # 모델 예측
        pred_probs = model.predict(padded, verbose=0)
        
        # 현재까지 입력된 시퀀스의 마지막 토큰 위치를 찾아 다음 토큰 예측
        current_last_index = len(encoded) - 1
        pred_id = np.argmax(pred_probs[0, current_last_index, :])
        
        encoded.append(pred_id)
        
        # EOS 토큰이 나오면 문장 생성을 정상적으로 종료
        if tokenizer.index_word.get(pred_id) == '<EOS>':
            break
            
    # 정수를 다시 문자열로 변환
    decoded_words = [tokenizer.index_word.get(i, '') for i in encoded]
    output_text = ' '.join(decoded_words).replace('<BOS>', '').replace('<EOS>', '').split('<SEP>')[-1].strip()
    return output_text

# 테스트 진행
test_input = "오늘 날씨 어때?"
result = generate_text(gpt_model, tokenizer, max_len, test_input)

print(f"질문: {test_input}")
print(f"GPT 답변 예측: {result}")

질문: 오늘 날씨 어때?
GPT 답변 예측: <bos> 오늘 날씨 어때 ? <sep> 너무 신경쓰지 마세요 . <eos>


## 6. 추가 분석: 챗봇 데이터셋 적용 시 표준 트랜스포머(Encoder-Decoder) 모델과의 비교

본 과제에서 구현한 GPT-1(Decoder-only) 구조와 기존의 표준 트랜스포머(Encoder-Decoder) 구조를 동일한 챗봇 데이터셋에 적용했을 때의 예상 결과와 아키텍처적 차이점은 다음과 같이 요약할 수 있습니다.

### 1) 학습 방식 및 구조적 메커니즘의 차이
* **표준 트랜스포머 (Encoder-Decoder):** * 질문(Q) 전체를 인코더가 읽고 문맥 정보를 압축한 뒤, 디코더가 이 정보를 넘겨받아 답변(A)을 생성하는 전형적인 Sequence-to-Sequence(시퀀스 투 시퀀스) 방식으로 동작합니다. 즉, 질문 해석(인코더)과 답변 생성(디코더)의 역할이 아키텍처 상에서 명확하게 분리되어 있습니다.
* **GPT 모델 (Decoder-only):** * 질문과 답변을 구분자(Delimiter)를 사용해 하나의 연속된 시퀀스(`질문 <SEP> 답변`)로 결합하여 입력합니다. 별도의 인코더 없이, 이전까지 입력된 모든 토큰을 바탕으로 '그다음 단어'를 예측하는 인과적 마스킹(Causal Masking) 기반의 Autoregressive(자기회귀) 방식으로 대화의 흐름을 학습합니다.

### 2) 대화 생성 결과물의 특징 비교
* **표준 트랜스포머의 결과:** * 인코더가 입력 문장 전체를 먼저 하이라이트하여 파악하므로, 일대일 매칭이 명확한 단답형 질의응답 환경에서 **질문의 목적과 의도에 엄격하게 부합하는 명확하고 안정적인 답변**을 생성하는 데 유리합니다.
* **GPT 모델의 결과:** * 문장 성분의 확률적 연결과 자연스러운 언어 텍스트 흐름을 모사하는 데 특화되어 있습니다. 따라서 단순한 정답 맞추기를 넘어, 이전 대화의 맥락을 유연하게 이어받거나 다채롭고 긴 문장을 막힘없이 이어나가는 자유 대화(Open-ended Generation) 능력에서 더 우수한 결과물을 보여줍니다.

### 3) 논문 기반 아키텍처 우수성 검증 (Ablation Study)
* 첨부된 논문의 절제 실험(Ablation Studies) 결과에 따르면, 동일한 프레임워크 내에서 트랜스포머 아키텍처 대신 기존의 순환신경망(LSTM) 구조를 적용했을 때 전반적인 자연어 이해 성능이 평균 **5.6점 하락**하는 것이 확인되었습니다.
* 이는 LSTM 구조가 단기 기억 의존성으로 인해 짧은 범위의 예측에 머무르는 반면, 트랜스포머 기반 구조는 **훨씬 더 장거리의 언어적 구조와 문맥(Longer-range linguistic structure)을 효과적으로 포착**할 수 있음을 증명합니다. 결과적으로 챗봇 데이터셋 학습 시에도 문맥이 길어질수록 GPT 아키텍처가 대화의 일관성을 유지하는 데 훨씬 강력한 성능을 발휘하게 됩니다.

# 원강(Won-gang) 프로젝트: 표준 트랜스포머(Encoder-Decoder) 기반 챗봇 모델 구현

## 1. 표준 트랜스포머 아키텍처 및 데이터 흐름의 변화

GPT-1(Decoder-Only) 구조에서 표준 트랜스포머(Encoder-Decoder) 구조로 변경 시 다음과 같은 핵심 아키텍처 및 데이터 전처리 변화가 발생합니다.

1. **데이터 분할 (인코더 입력 vs 디코더 입력)**:
   * **GPT**: `[<BOS> 질문 <SEP> 답변 <EOS>]` 구조의 단일 시퀀스로 학습.
   * **트랜스포머**: 
     * 인코더 입력(Encoder Input): `[질문]`
     * 디코더 입력(Decoder Input): `[<BOS> 답변]`
     * 디코더 타겟(Decoder Target): `[답변 <EOS>]`

2. **인코더-디코더 상호작용 (Cross-Attention)**:
   * 디코더 블록 내부에 **Multi-Head Cross-Attention** 레이어가 추가됩니다.
   * 이 레이어는 디코더의 현재 상태(Query)가 인코더가 분석한 질문의 정보(Key, Value) 중 어떤 부분에 집중해야 하는지(Attention) 연결해 주는 역할을 합니다.

In [9]:
import urllib.request
import pandas as pd
import tensorflow as tf
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 데이터 로드
urllib.request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatbotData.csv")
train_data = pd.read_csv('ChatbotData.csv')
train_data = train_data[:2000] # 샘플링

# 토큰 정의
BOS, EOS = "<BOS>", "<EOS>"

def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    return sentence.strip()

encoder_inputs, decoder_inputs, decoder_targets = [], [], []

for _, row in train_data.iterrows():
    q = preprocess_sentence(row['Q'])
    a = preprocess_sentence(row['A'])
    
    # 표준 트랜스포머 형태에 맞게 데이터 분리
    encoder_inputs.append(q)
    decoder_inputs.append(f"{BOS} {a}")
    decoder_targets.append(f"{a} {EOS}")

# 토크나이징 (인코더와 디코더 통합 토크나이저 사용)
tokenizer = Tokenizer(filters='', oov_token='<OOV>')
tokenizer.fit_on_texts(encoder_inputs + decoder_inputs + decoder_targets)
vocab_size = len(tokenizer.word_index) + 1

# 정수 인코딩 및 패딩
X_enc = pad_sequences(tokenizer.texts_to_sequences(encoder_inputs), padding='post')
X_dec = pad_sequences(tokenizer.texts_to_sequences(decoder_inputs), padding='post')
Y_dec = pad_sequences(tokenizer.texts_to_sequences(decoder_targets), padding='post')

max_enc_len = X_enc.shape[1]
max_dec_len = X_dec.shape[1]

print(f"Vocab Size: {vocab_size}")
print(f"Encoder Input Shape: {X_enc.shape}")
print(f"Decoder Input Shape: {X_dec.shape}")

Vocab Size: 4655
Encoder Input Shape: (2000, 9)
Decoder Input Shape: (2000, 15)


In [11]:
# 과제 3 & 4: 인코더, 디코더 및 전체 트랜스포머 모델 구성 - (KerasTensor 오류 수정본)
import tensorflow as tf

# 패딩 및 마스킹 헬퍼 함수
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    return tf.tile(tf.reshape(tf.cast(i >= j, dtype), [1, n_dest, n_src]), [batch_size, 1, 1])

# 위치 임베딩 레이어 (학습 가능한 Embedding)
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(PositionalEmbedding, self).__init__()
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(max_len, embed_dim)

    def call(self, x):
        max_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

# 인코더 블록 (Self-Attention만 존재)
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads, embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='gelu'),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        attn_output = self.mha(x, x) # Self-Attention
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# 디코더 블록 (Self-Attention + Cross-Attention)
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(DecoderLayer, self).__init__()
        self.mha1 = tf.keras.layers.MultiHeadAttention(num_heads, embed_dim) # Masked Self
        self.mha2 = tf.keras.layers.MultiHeadAttention(num_heads, embed_dim) # Cross-Attention
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='gelu'),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, enc_output, training=False):
        # [수정된 부분] 밖에서 계산하던 마스크를 레이어 내부로 안전하게 이동
        input_shape = tf.shape(x)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        look_ahead_mask = causal_attention_mask(batch_size, seq_len, seq_len, tf.bool)
        
        # 1. Masked Self-Attention
        attn1 = self.mha1(x, x, attention_mask=look_ahead_mask)
        out1 = self.layernorm1(x + attn1)
        
        # 2. Cross-Attention (Query는 디코더[out1], Key/Value는 인코더 출력[enc_output])
        attn2 = self.mha2(out1, enc_output, enc_output)
        out2 = self.layernorm2(out1 + attn2)
        
        # 3. Feed Forward
        ffn_output = self.ffn(out2)
        return self.layernorm3(out2 + ffn_output)

# 하이퍼파라미터 설정
embed_dim = 128
num_heads = 4
ff_dim = 256

# 빌드: Encoder-Decoder 함수형 API 연결
enc_inputs = tf.keras.layers.Input(shape=(None,), dtype=tf.int32, name="enc_inputs")
dec_inputs = tf.keras.layers.Input(shape=(None,), dtype=tf.int32, name="dec_inputs")

# 인코더 흐름
enc_embed = PositionalEmbedding(max_enc_len, vocab_size, embed_dim)(enc_inputs)
enc_output = EncoderLayer(embed_dim, num_heads, ff_dim)(enc_embed)

# 디코더 흐름 (수정됨: 마스크 계산 없이 바로 연결)
dec_embed = PositionalEmbedding(max_dec_len, vocab_size, embed_dim)(dec_inputs)
dec_output = DecoderLayer(embed_dim, num_heads, ff_dim)(dec_embed, enc_output)

final_outputs = tf.keras.layers.Dense(vocab_size, activation="softmax")(dec_output)

# 전체 트랜스포머 모델 정의
transformer_model = tf.keras.Model(inputs=[enc_inputs, dec_inputs], outputs=final_outputs)
transformer_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

transformer_model.summary()

# 학습 실행
print("표준 트랜스포머 모델 학습을 시작합니다...")
transformer_model.fit([X_enc, X_dec], Y_dec, batch_size=32, epochs=5)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_inputs          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_inputs          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    596,992 │ enc_inputs[0][0]  │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    597,760 │ dec_inputs[0][0]  │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_layer_1     │ (None, None, 128) │    330,240 │ positional_embed… │
│ (EncoderLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer       │ (None, None, 128) │    594,304 │ positional_embed… │
│ (DecoderLayer)      │                   │            │ encoder_layer_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, None,      │    600,495 │ decoder_layer[0]… │
│                     │ 4655)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,719,791 (10.38 MB)

 Trainable params: 2,719,791 (10.38 MB)

 Non-trainable params: 0 (0.00 B)

표준 트랜스포머 모델 학습을 시작합니다...
Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 29s 295ms/step - loss: 3.3307
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 301ms/step - loss: 1.5432
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 293ms/step - loss: 1.2218
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 298ms/step - loss: 0.9821
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 293ms/step - loss: 0.7951


In [12]:
# 추론(Inference) 단계: 인코더를 한 번 거친 뒤 디코더가 루프를 돌며 예측합니다.
def evaluate_transformer(input_sentence):
    q = preprocess_sentence(input_sentence)
    enc_input = pad_sequences(tokenizer.texts_to_sequences([q]), maxlen=max_enc_len, padding='post')
    
    # 디코더의 시작은 오직 <BOS> 토큰만 존재합니다.
    dec_input = tokenizer.texts_to_sequences([BOS])
    
    for _ in range(max_dec_len):
        padded_dec = pad_sequences(dec_input, maxlen=max_dec_len, padding='post')
        predictions = transformer_model.predict([enc_input, padded_dec], verbose=0)
        
        # 현재 루프 단계의 마지막 단어 위치 추출
        current_idx = len(dec_input[0]) - 1
        predicted_id = np.argmax(predictions[0, current_idx, :])
        
        dec_input[0].append(predicted_id)
        
        # 문장 끝 토큰을 만나면 종료
        if tokenizer.index_word.get(predicted_id) == '<EOS>':
            break
            
    decoded_words = [tokenizer.index_word.get(i, '') for i in dec_input[0]]
    return ' '.join(decoded_words).replace(BOS, '').replace(EOS, '').strip()

# 결과 출력
test_q = "오늘 날씨 어때?"
print(f"질문: {test_q}")
print(f"트랜스포머 답변 결과: {evaluate_transformer(test_q)}")

질문: 오늘 날씨 어때?
트랜스포머 답변 결과: <bos> 새로운 스타일 도전해 보시면 어때요 ? <eos> 해보세요 . <eos>
